# Bend Topology Optimization -- Fabrication-Bias-Robust Adjoint Design (Compact Domain, 3um Region, +/-10nm)

This notebook builds a fabrication-bias-robust adjoint topology optimization,
comparing against a fresh, non-Euler-warm-started **baseline found on this
same compact domain** -- NOT notebook 04's own design, which was already
Euler-warm-started and therefore already somewhat robust "by luck," which
would confound the comparison.

Compared to an earlier revision of this notebook: the simulation cell is now
a tight, `bend.py`-style asymmetric box (eliminating ~36% of wasted cell
area), the design region is 3x3um (down from 4.5x4.5um), the fabrication-bias
target is +/-10nm (up from +/-5nm, which proved too small an effect to
measure meaningfully), and the comparison baseline is a fresh, non-Euler
design found by a 5-random-seed search on this compact domain, gated at
>90% transmission before anything downstream is trusted.

**Do not use "Run All."** Section 5 (5-seed baseline search) and Section 11
(robust optimization) run real, expensive FDTD.

In [ ]:
import inspect
import json

import numpy as np
import matplotlib.pyplot as plt

from pic_toolkit.meep_sim import bend_topopt
from pic_toolkit.meep_sim import bend_topopt_robust as btr
from pic_toolkit import checks, viz, sparams, design_points, style

style.apply_style()
%matplotlib inline

## 1. Recap: why the domain and the comparison baseline changed

`bend_topopt.py` (notebook 04) builds its simulation cell with a UNIFORM
clearance around the design region's bounding box on all 4 sides. Since the
bend only has arms on 2 of those sides, the other 2 sides carry no structure
at all -- measured directly: ~32% of that cell's 86.24um^2 area is empty.
`bend.py` (notebook 02) avoids this by sizing its cell to the L-shaped
structure's own tight bounding box, asymmetric on purpose: minimal clearance
on the two "empty" sides, full PML+port reach only on the two sides that
carry an arm. Section 2 below applies that same idea here.

Separately: notebook 04's own optimized design was warm-started from a
smooth Euler-spiral curve (`build_euler_initial_density`), not from a blank
or random start. A smooth, already-good starting shape is itself a source
of accidental robustness -- comparing THIS notebook's robust design against
notebook 04 would conflate "the 3-way averaged objective helped" with "the
warm start already helped." Section 5 instead searches for a non-Euler
baseline directly on this notebook's own compact domain, and Section 8
measures THAT baseline's own fabrication-bias sensitivity as the "before"
number the robust design (Section 16) is compared against.

## 2. The compact domain

`_domain_compact()` mirrors `bend.py`'s own `_domain()`: the cell is sized to
the structure's actual footprint plus the minimum necessary PML/margin
clearance, not a fixed symmetric box. The input (horizontal) arm reaches from
the cell's left edge to the design region's right edge; the output (vertical)
arm reaches from the design region's bottom edge to the cell's top edge. The
two "arm" sides need `dpml_um + source_inset_um + port_gap_um + arm_lead_um`
of reach; the two "empty" sides need only `dpml_um + margin_um`:

```
reach_arm   = dpml_um + source_inset_um + port_gap_um + arm_lead_um
reach_empty = dpml_um + margin_um
cell_side   = reach_arm + design_region_x_um + reach_empty
```

`arm_lead_um` (new) is a small, deliberate buffer between the port reference
plane and the design region's own edge -- in the OLD domain, an equivalent
buffer existed too, just as an unintentional ~1.65um byproduct of the
symmetric-clearance box, which is exactly what made that box ~32% wasted
area.

In [ ]:
params = dict(btr.DEFAULT_PARAMS)  # this module's own working copy
dom = btr._domain_compact(params)

old_area = 9.8 * 8.8  # bend_topopt.py's own cell_x_um * cell_y_um
new_area = dom["cell_x_um"] * dom["cell_y_um"]
print(f"old (bend_topopt.py) cell area: {old_area:.2f} um^2")
print(f"new (compact) cell area:        {new_area:.2f} um^2")
print(f"reduction: {(old_area - new_area) / old_area:.1%}")
print()
for k, v in dom.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

In [ ]:
# Blank (init_density=0.5) permittivity map with port annotations -- confirm
# the compact domain places everything where expected before any real FDTD.
blank_perm = btr.get_permittivity_map_compact(params, weights=None)
ports = [
    (dom["port1_x"], dom["y0"], "o1", "v"),
    (dom["x0"], dom["port2_y"], "o2", "h"),
]
fig, ax = plt.subplots(figsize=(6, 6))
viz.plot_permittivity(blank_perm.eps, blank_perm.extent_um, ax=ax, ports=ports, pml_um=params["dpml_um"])
ax.set_title("Compact domain -- blank (pre-optimization) design region")
plt.show()

> **STOP -- visually confirm the ports, arms, and design region sit where
> expected (design region corner at bottom-right of the design region's own
> footprint, arms reaching the correct cell edges, PML shading not
> overlapping any port) before running any real FDTD.**

## 3. Design-grid resolution for the 3um region -- re-verified, not assumed

The earlier revision of this module found that `mpa.conic_filter` evaluates
its own kernel only at the design grid's own sample points (no internal
upsampling) -- at `bend_topopt.py`'s `design_grid_n=31` (150nm pitch) and
`filter_radius_um=0.2`, the filter's transition zone spanned barely one grid
cell, and at the schedule's sharpest stage (`beta=32`) the eta-threshold
mechanism this whole notebook depends on went completely dead (measured:
exactly zero boundary shift across a wide eta range). The fix there was
`design_grid_n=181` (25nm pitch). This module's own `design_grid_n=121`
keeps that same 25nm pitch at the new 3.0um region size -- but "the pitch
matches" is not proof it still works; it must be re-measured here, at
`beta=32`, before trusting anything built on it.

In [ ]:
etas = np.linspace(0.35, 0.65, 21)
bias_check_um = btr.calibrate_eta_bias(params, beta=32.0, eta_candidates=etas)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(etas, 1000 * bias_check_um, "-o", ms=3, color=style.COLOR_STEEL)
ax.axhline(0, color=style.COLOR_REFERENCE, linestyle="--", linewidth=1)
ax.set_xlabel("eta")
ax.set_ylabel("measured boundary bias (nm)")
ax.set_title(f"design_grid_n={params['design_grid_n']} eta<->nm check at beta=32")
plt.show()

is_degenerate = np.allclose(bias_check_um, 0.0, atol=1e-4)
print(f"design_grid_n={params['design_grid_n']}: {'DEGENERATE (all ~0) -- escalate design_grid_n!' if is_degenerate else 'non-degenerate, looks usable'}")
print(f"measured range: {1000*bias_check_um.min():.2f} to {1000*bias_check_um.max():.2f} nm over eta in [{etas.min()},{etas.max()}]")

> **STOP -- if the curve above is degenerate (near-zero bias across a wide
> eta range, the same failure mode `design_grid_n=31` showed at the old
> region), escalate `design_grid_n` (try 161, 201) and re-run this section
> before proceeding. Do not continue with a dead eta<->nm mechanism.**

## 4. Calibrating `eta_e`/`eta_d` against a physical ±10nm bias

**So, what is this optimization actually doing?** Rather than tuning a small
number of parameters (like a bend radius), this notebook divides the design
region into a fine grid of `design_grid_n` x `design_grid_n` cells and lets
the optimizer choose, independently for EVERY cell, whether it should be
silicon or cladding (each cell's value runs continuously from 0=cladding to
1=silicon during the search) -- whatever shape maximizes transmission.

**Why not let the optimizer choose completely freely?** Left alone, a
gradient-based optimizer tends to produce a checkerboard-like pattern of
tiny, isolated pixels. Such a pattern can look good in the simulation, but
no real fabrication process (photolithography) can print features that
small -- there's a hard physical limit on the smallest linewidth/gap it can
resolve. So a pattern that is only "good on paper" is not actually useful.

**The conic filter (`mpa.conic_filter`) -- a "blur," exactly like in image
editing software.** To prevent that, every cell's value is averaged with its
neighbors before being evaluated -- neighbors within a radius `R`
(`filter_radius_um`, 0.2um here) count more the closer they are, fading
linearly to zero weight at distance `R` (a cone-shaped weighting, hence
"conic"). This radius `R` effectively becomes the smallest feature size the
final design can have -- anything narrower than `R` gets blurred away.

**The tanh projection (`mpa.tanh_projection`) -- pushing back to black/white.**
After blurring, a cell's value is a gray, in-between number, but a real,
buildable structure needs a hard yes/no answer (silicon or not) at each
point. This second step pushes the blurred, gray value back toward 0 or 1,
using a threshold `eta`: values above `eta` get pushed toward 1 (silicon),
values below toward 0 (cladding). `beta` controls how sharp that push is.

**Why this two-step process is exactly what's needed to model fabrication
error.** At `eta=0.5` (the "as-designed" setting, called `eta_i` here),
nothing is biased either way. Push `eta` UP, and it takes a larger blurred
value to still count as silicon -- so silicon shrinks slightly everywhere
its boundary is. That is precisely what happens in a real fabrication
process when etching removes a little too much material (under-sizing /
**erosion**). Push `eta` DOWN and silicon grows instead (over-sizing /
**dilation**). So, by moving one single number, `eta`, we can simulate "what
if this exact design came out of the fab a little too thick / a little too
thin" -- without building or simulating a different structure by hand.

**What the plot below measures, and what its axes mean.** To find exactly
which `eta` corresponds to a real ±10nm shift, `calibrate_eta_bias` builds a
simple test pattern (silicon on one side of a straight line, cladding on the
other), runs it through the SAME blur+threshold steps used everywhere else,
renders it into an actual Meep `mp.MaterialGrid` (so it goes through the
exact same rendering the real simulation uses), and measures where the
silicon/cladding boundary actually lands.

- **x-axis**: the candidate `eta` value (swept near 0.5).
- **y-axis**: how far the boundary has moved (in nm) relative to `eta=0.5`,
  positive meaning "silicon grew" (dilation). Call this `bias_um(eta)`.

**`eta_e` and `eta_d`, precisely.** These are simply the two points on that
curve where the shift equals the target:

```
eta_d  solves  bias_um(eta_d) = +target_bias_um   (silicon grows by 10nm)
eta_e  solves  bias_um(eta_e) = -target_bias_um   (silicon shrinks by 10nm)
```

found automatically by `find_eta_for_bias` (`scipy.optimize.brentq`, a
standard "find where this curve crosses a target value" root-finder -- no
new theory, just automating what could otherwise be read off the plot by
eye). The optimizer below then evaluates the SAME design three times every
iteration -- once at `eta_i` (as designed), once at `eta_e` (10nm thinner),
once at `eta_d` (10nm thicker) -- and maximizes the AVERAGE of the three
transmissions, pushing the final shape toward one that still works well even
if the real, fabricated part comes out a little thicker or thinner than
drawn.

**(For reference, the equations behind the two steps above:)**

```
filtered(cell) = sum(neighbor_value * weight) / sum(weight),
                 weight = max(0, R - distance_to_neighbor)      <- conic filter

projected = [tanh(beta*eta) + tanh(beta*(filtered - eta))]
          / [tanh(beta*eta) + tanh(beta*(1 - eta))]             <- tanh projection

bias_um(eta) = -(position(eta) - position(0.5))                <- calibration
```
Nothing here needs to be memorized -- "blur, then threshold, then slide the
threshold to fake a fabrication error" is the whole idea.

In [ ]:
etas_fine = np.linspace(0.35, 0.65, 31)
bias_curve_um = btr.calibrate_eta_bias(params, beta=32.0, eta_candidates=etas_fine)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(etas_fine, 1000 * bias_curve_um, "-o", ms=3, color=style.COLOR_STEEL)
ax.axhline(0, color=style.COLOR_REFERENCE, linestyle="--", linewidth=1)
ax.axhline(1000 * params["target_bias_um"], color=style.COLOR_ANNOTATION, linestyle=":", linewidth=1)
ax.axhline(-1000 * params["target_bias_um"], color=style.COLOR_ANNOTATION, linestyle=":", linewidth=1)
ax.set_xlabel("eta (tanh-projection threshold)")
ax.set_ylabel("boundary bias (nm, + = dilation / Si grows)")
ax.set_title("Measured eta -> physical bias calibration (beta=32)")
plt.show()

eta_d = btr.find_eta_for_bias(params, beta=32.0, target_bias_um=+params["target_bias_um"])
eta_e = btr.find_eta_for_bias(params, beta=32.0, target_bias_um=-params["target_bias_um"])
print(f"eta_i (nominal) = {params['eta_i']}")
print(f"eta_d (dilation, +{1000*params['target_bias_um']:.0f}nm) = {eta_d:.5f}")
print(f"eta_e (erosion,  -{1000*params['target_bias_um']:.0f}nm) = {eta_e:.5f}")

params["eta_e"], params["eta_d"] = eta_e, eta_d

In [ ]:
# Does the calibration depend on beta? Measured, not assumed.
eta_d_beta4 = btr.find_eta_for_bias(params, beta=4.0, target_bias_um=+params["target_bias_um"])
eta_e_beta4 = btr.find_eta_for_bias(params, beta=4.0, target_bias_um=-params["target_bias_um"])
print(f"beta=4.0:  eta_e={eta_e_beta4:.5f}  eta_d={eta_d_beta4:.5f}")
print(f"beta=32.0: eta_e={eta_e:.5f}  eta_d={eta_d:.5f}")
print(f"eta_e delta: {abs(eta_e_beta4 - eta_e):.5f}   eta_d delta: {abs(eta_d_beta4 - eta_d):.5f}")

> **STOP -- inspect the calibration curve and the beta=4 vs. beta=32
> comparison above before proceeding.** If `eta_e`/`eta_d` disagree
> meaningfully across beta, a per-beta-stage lookup should replace the
> single fixed pair used below.

## 5. Baseline search: 5 corner-biased random initial guesses (non-Euler, single-objective, compact domain)

**A first attempt at this baseline search used PURE per-pixel uniform random
noise as the initial density, and failed** -- none of 5 seeds reached even
30% transmission. Measured directly (before re-attempting): pure i.i.d.
noise collapses through `mpa.conic_filter` to a near-constant ~0.5 field
(its standard deviation drops ~10x), and `mpa.tanh_projection` then
re-fragments that tiny residual into a spatially-UNCORRELATED binary
texture -- effectively as uninformative a starting point as the
already-documented-to-fail blank/uniform-0.5 start (see
`docs/troubleshooting_log.md`), just noisier-looking.

Instead, `build_corner_biased_random_density` makes each pixel's silicon
PROBABILITY a Gaussian function of its distance from the sharp (radius=0)
90-degree corner path connecting the two ports -- concentrating density
near a physically sensible path while still drawing every pixel's own value
independently at random (real seed-to-seed diversity, not a fixed shape;
this is NOT the same as `bend.py`'s deterministic sharp-corner geometry, nor
the smooth Euler curve `build_euler_initial_density` rasterizes -- it is a
genuinely random field whose local MEAN follows that Gaussian envelope).

In [ ]:
corner_xy = (dom["x0"], dom["y0"])
sigma_um = params["baseline_corner_bias_sigma_um"]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, seed in zip(axes, [0, 1, 2]):
    d = btr.build_corner_biased_random_density(params, seed=seed, sigma_um=sigma_um, corner_xy=corner_xy)
    im = ax.imshow(d.T, origin="lower", cmap=style.CMAP_DENSITY, vmin=0, vmax=1)
    ax.set_title(f"seed={seed}  fill={d.mean():.3f}")
fig.suptitle(f"Corner-biased random initial densities (sigma_um={sigma_um})")
plt.show()

> **STOP -- confirm the density maps above show real, informative
> structure (denser near the corner, sparser away from it, but still
> genuinely random pixel-to-pixel) rather than near-uniform gray or
> incoherent salt-and-pepper noise -- the same failure mode Section 5's own
> markdown just described for pure random noise. If it still looks
> degenerate, adjust `baseline_corner_bias_sigma_um` and re-run this cell.**

> **STOP -- this runs 5 independent single-objective adjoint optimizations
> (roughly 5x one training run's own cost) in the compact domain, none
> warm-started from the Euler curve. Confirm before running.**

In [ ]:
baseline_params = dict(params)
n = baseline_params["design_grid_n"]
seeds = [0, 1, 2, 3, 4]
baseline_candidates = {}

for seed in seeds:
    init = btr.build_corner_biased_random_density(baseline_params, seed=seed, sigma_um=sigma_um, corner_xy=corner_xy)
    opt_result = btr.run_adjoint_optimization_compact(baseline_params, init_weights=init)
    val = btr.simulate_baseline_compact(baseline_params, weights=opt_result.final_weights_binarized)
    i_mid = len(val.s_matrix["21"]) // 2
    T21 = float(np.abs(val.s_matrix["21"][i_mid]) ** 2)
    baseline_candidates[seed] = {"opt_result": opt_result, "validation": val, "T21": T21}
    print(f"seed={seed}: T21={T21:.4f}")

## 6. Baseline candidates' honest transmission -- convergence check, then the 90% gate

Before trusting (or distrusting) the 90% gate below, check whether
`iters_per_stage`/`beta_schedule`'s 80-iteration budget was actually enough
-- i.e. was each seed's objective still climbing when the budget ran out, or
had it plateaued? A saturated curve well below the gate means "this budget
is enough, but this design/start can't reach 90%"; a still-climbing curve
means "more iterations might still help" -- these call for different next
steps.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for seed, color in zip(seeds, style.COLOR_CYCLE):
    hist = baseline_candidates[seed]["opt_result"].evaluation_history
    ax.plot(np.arange(1, len(hist) + 1), hist, "-o", ms=3, label=f"seed={seed}", color=color)
for stage_end in range(params["iters_per_stage"], len(baseline_candidates[seeds[0]]["opt_result"].evaluation_history), params["iters_per_stage"]):
    ax.axvline(stage_end + 0.5, color=style.COLOR_REFERENCE, linestyle="--", linewidth=1, alpha=0.6)
ax.set_xlabel("NLopt iteration")
ax.set_ylabel("objective J")
ax.set_title("Baseline search convergence -- has J saturated by iteration 80?")
ax.legend()
plt.show()

# Quantify: how much did J change over the LAST beta stage (iterations 61-80)
# vs. the stage before it (41-60)? A ratio near 0 means "saturated"; a ratio
# still near 1 means "still improving as fast as ever when the budget ran out."
for seed in seeds:
    hist = baseline_candidates[seed]["opt_result"].evaluation_history
    stage = params["iters_per_stage"]
    delta_last = hist[-1] - hist[-stage]
    delta_prev = hist[-stage] - hist[-2 * stage] if len(hist) >= 2 * stage else float("nan")
    ratio = abs(delta_last) / abs(delta_prev) if delta_prev else float("nan")
    print(f"seed={seed}: J[-2*stage]={hist[-2*stage]:.4f} J[-stage]={hist[-stage]:.4f} J[-1]={hist[-1]:.4f}  "
          f"last-stage-delta={delta_last:+.4f}  saturation_ratio={ratio:.2f}")

In [ ]:
transmissions = {s: baseline_candidates[s]["T21"] for s in seeds}
print(json.dumps(transmissions, indent=2))

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar([str(s) for s in seeds], [transmissions[s] for s in seeds], color=style.COLOR_STEEL)
ax.axhline(params["baseline_transmission_gate"], color=style.COLOR_ANNOTATION, linestyle="--",
           label=f"gate = {params['baseline_transmission_gate']:.0%}")
ax.set_xlabel("random seed")
ax.set_ylabel(r"transmission $|S_{21}|^2$")
ax.set_title("5-corner-biased-random-seed baseline search")
ax.legend()
plt.show()

best_seed = max(seeds, key=lambda s: transmissions[s])
assert max(transmissions.values()) > params["baseline_transmission_gate"], (
    f"None of the {len(seeds)} corner-biased-random-start baseline designs reached "
    f"{params['baseline_transmission_gate']:.0%} transmission "
    f"(measured per-seed: {transmissions}). STOP HERE -- do not proceed to the "
    f"resolution check / robust-optimization sections below. Consult the user "
    f"before changing anything -- do not silently lower this gate, enlarge the "
    f"design region, or add a warm start."
)
print(f"\nGate passed. best_seed={best_seed}, T21={transmissions[best_seed]:.4f}")
baseline_result = baseline_candidates[best_seed]["opt_result"]

## 7. Resolution-adequacy check for the ±10nm transmission delta

Is `resolution=20` (50nm Yee pixels) fine enough to reliably measure a
10nm-scale transmission change, or is `resolution=40` needed for the
validation calls below (the training loop itself stays at 20 regardless --
see the note in the code below)?

In [ ]:
baseline_mask = baseline_result.final_weights_binarized  # (n,n), n=params["design_grid_n"] already
resolution_check = {}
for res in (20, 40):
    res_params = {**baseline_params, "resolution": res}
    r_nom = btr.simulate_baseline_compact(res_params, weights=baseline_mask)
    r_ero = btr.simulate_baseline_compact(res_params, weights=btr._mapping_robust(
        baseline_mask.flatten(), {**res_params, "beta": 64.0}, params["eta_e"]).reshape(n, n))
    r_dil = btr.simulate_baseline_compact(res_params, weights=btr._mapping_robust(
        baseline_mask.flatten(), {**res_params, "beta": 64.0}, params["eta_d"]).reshape(n, n))

    i_mid = len(r_nom.s_matrix["21"]) // 2
    t_nom = float(np.abs(r_nom.s_matrix["21"][i_mid]) ** 2)
    t_ero = float(np.abs(r_ero.s_matrix["21"][i_mid]) ** 2)
    t_dil = float(np.abs(r_dil.s_matrix["21"][i_mid]) ** 2)
    resolution_check[res] = {"nominal": t_nom, "eroded": t_ero, "dilated": t_dil}
    print(f"resolution={res}: T_nominal={t_nom:.4f}  T_eroded={t_ero:.4f}  T_dilated={t_dil:.4f}  "
          f"max|delta|={max(abs(t_ero-t_nom), abs(t_dil-t_nom)):.4f}")

In [ ]:
# Same rule as before: keep resolution=20 inside the (expensive) training loop
# regardless of this check's outcome -- 2D FDTD cost scales ~resolution^3, so
# doubling it there would multiply the already-3x-inflated per-iteration cost
# roughly another 8x. resolution=40 (if needed) is only used for these cheap,
# one-off validation calls.
delta_20 = max(abs(resolution_check[20]["eroded"] - resolution_check[20]["nominal"]),
               abs(resolution_check[20]["dilated"] - resolution_check[20]["nominal"]))
delta_40 = max(abs(resolution_check[40]["eroded"] - resolution_check[40]["nominal"]),
               abs(resolution_check[40]["dilated"] - resolution_check[40]["nominal"]))
relative_change = abs(delta_40 - delta_20) / delta_20 if delta_20 > 0 else float("inf")
resolution_converged = relative_change < 0.15
chosen_resolution = 20 if resolution_converged else 40

print(f"transmission-delta at resolution=20: {delta_20:.4f}")
print(f"transmission-delta at resolution=40: {delta_40:.4f}")
print(f"relative change: {relative_change:.1%}")
print(f"CONCLUSION: resolution={'20 is adequate' if resolution_converged else '40 is required'} "
      f"for the fabrication-bias validation numbers reported below.")

## 8. Baseline design's fabrication-bias sensitivity at ±10nm (the new "before" number)

In [ ]:
baseline_fab_tolerance = resolution_check[chosen_resolution]
print(f"baseline (best-of-5-random-seed) design, at resolution={chosen_resolution}:")
print(f"  T(nominal) = {baseline_fab_tolerance['nominal']:.4f}  "
      f"({-10*np.log10(baseline_fab_tolerance['nominal']):.3f} dB)")
print(f"  T(+{1000*params['target_bias_um']:.0f}nm dilated) = {baseline_fab_tolerance['dilated']:.4f}  "
      f"({-10*np.log10(baseline_fab_tolerance['dilated']):.3f} dB)")
print(f"  T(-{1000*params['target_bias_um']:.0f}nm eroded)  = {baseline_fab_tolerance['eroded']:.4f}  "
      f"({-10*np.log10(baseline_fab_tolerance['eroded']):.3f} dB)")
max_delta_baseline = max(abs(baseline_fab_tolerance["dilated"] - baseline_fab_tolerance["nominal"]),
                          abs(baseline_fab_tolerance["eroded"] - baseline_fab_tolerance["nominal"]))
print(f"  max|delta T| = {max_delta_baseline:.4f}")

baseline_res_params = {**baseline_params, "resolution": chosen_resolution}
eroded_baseline_mask = btr._mapping_robust(baseline_mask.flatten(), {**baseline_res_params, "beta": 64.0}, params["eta_e"]).reshape(n, n)
dilated_baseline_mask = btr._mapping_robust(baseline_mask.flatten(), {**baseline_res_params, "beta": 64.0}, params["eta_d"]).reshape(n, n)

eroded_perm = btr.get_permittivity_map_compact(baseline_res_params, weights=eroded_baseline_mask)
nominal_perm = btr.get_permittivity_map_compact(baseline_res_params, weights=baseline_mask)
dilated_perm = btr.get_permittivity_map_compact(baseline_res_params, weights=dilated_baseline_mask)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
viz.plot_permittivity(eroded_perm.eps, eroded_perm.extent_um, ax=axes[0], cmap=style.CMAP_PERMITTIVITY_BASELINE)
axes[0].set_title(f"Eroded (-{1000*params['target_bias_um']:.0f}nm)")
viz.plot_permittivity(nominal_perm.eps, nominal_perm.extent_um, ax=axes[1], cmap=style.CMAP_PERMITTIVITY_BASELINE)
axes[1].set_title("Baseline (best of 5 random seeds)")
viz.plot_permittivity(dilated_perm.eps, dilated_perm.extent_um, ax=axes[2], cmap=style.CMAP_PERMITTIVITY_BASELINE)
axes[2].set_title(f"Dilated (+{1000*params['target_bias_um']:.0f}nm)")
fig.tight_layout()
plt.show()

> **STOP -- visually inspect all three permittivity maps for any vanished,
> merged, or unexpectedly-thinned feature before trusting the transmission
> numbers above.**

## 9. Euler warm start for the 3um region -- re-verified, not assumed

`build_euler_initial_density`'s curve is anchored at the design region's own
center (= the bend's corner point), so it only ever occupies ONE quadrant of
the design region -- the real constraint is footprint (~1.87 * radius_um)
`<=` `design_region_x_um / 2`, not the full `design_region_x_um`. Confirmed
directly during this notebook's own construction: `radius_um=1.0` (which
fits comfortably in `bend_topopt.py`'s own 4.5um region) raises `ValueError`
here at 3.0um, precisely because `Lx/2=1.5um < footprint=1.87um`. The value
below (`init_bend_radius_um=0.6`, footprint 1.122um, margin 0.378um) was
chosen to match notebook 04's own original margin ratio, not assumed safe.

In [ ]:
design_region_center = (dom["x0"], dom["y0"])
euler_params = {**params, "design_region_center": design_region_center}
initial_density = bend_topopt.build_euler_initial_density(euler_params, params["init_bend_radius_um"])

perm_map = btr.get_permittivity_map_compact(params, weights=initial_density)
fig, ax = plt.subplots(figsize=(6, 6))
viz.plot_permittivity(perm_map.eps, perm_map.extent_um, ax=ax, ports=ports, pml_um=params["dpml_um"])
ax.set_title(f"Euler warm start (radius={params['init_bend_radius_um']}um) in the compact 3um domain")
plt.show()
print(f"initial density fill fraction: {initial_density.mean():.3f}")

> **STOP -- if `build_euler_initial_density` raised `ValueError`, or the
> rasterized curve above looks clipped/discontinuous, reduce
> `init_bend_radius_um` and re-run before proceeding to Section 11's
> expensive optimization.**

## 10. The three-way averaged-objective formulation

`build_optimization_problem_compact` has no dependency on `eta` anywhere in
its own construction -- `eta` only enters downstream, in `_mapping_robust`,
when deciding what density array to feed the already-built
`OptimizationProblem`. So three INDEPENDENT calls to that same function
(`build_robust_optimization_problem_compact`) give three structurally
identical `mp.Simulation`/`MaterialGrid`/`DesignRegion` triples that differ
only in which `_mapping_robust(..., eta)` rendering of the SAME raw design
variables `x` is fed into each one's `opt(...)` call:

$$J_\text{avg}(x) = \frac{1}{3}\Big[J\big(\text{map}(x,\eta_i)\big) + J\big(\text{map}(x,\eta_e)\big) + J\big(\text{map}(x,\eta_d)\big)\Big]$$

Because differentiation is linear, $\nabla_x J_\text{avg} = \frac{1}{3}
\big[\nabla_x J_n + \nabla_x J_e + \nabla_x J_d\big]$ exactly -- real
adjoint gradients flow through all three branches independently; NLopt/MMA
is handed one averaged scalar and one averaged gradient vector, at the cost
of three forward+adjoint FDTD solves per iteration instead of one.

In [ ]:
print(inspect.getsource(btr._mapping_robust))
print(inspect.getsource(btr.build_robust_optimization_problem_compact))
print(inspect.getsource(btr.run_robust_adjoint_optimization_compact))

## 11. Run the robust adjoint optimization

> **STOP -- this cell runs the full 3-way robust adjoint optimization loop:
> three forward + three adjoint FDTD runs per NLopt iteration, across all of
> `beta_schedule`'s stages. Expect roughly 3x a single-objective run's own
> runtime -- run headlessly with a generous `nbconvert` timeout, and wait.**

In [ ]:
robust_result = btr.run_robust_adjoint_optimization_compact(params, init_weights=initial_density)
print(f"\n{len(robust_result.evaluation_history_avg)} iterations across "
      f"{len(params['beta_schedule'])} beta stages, "
      f"J_avg: {robust_result.evaluation_history_avg[0]:.4f} -> {robust_result.evaluation_history_avg[-1]:.4f}")
print(f"final design fill fraction (nominal): {robust_result.final_weights_binarized['nominal'].mean():.3f}")

## 12. Convergence

Plotted here are all three sub-objectives individually plus their average
(the quantity NLopt actually maximizes). A dip at a beta-stage boundary
(discontinuous re-projection of the carried-over raw variable at a sharper
threshold) is expected, not a bug.

In [ ]:
iters_per_stage = params["iters_per_stage"]
fig, ax = plt.subplots(figsize=(8, 4.5))
iterations = np.arange(1, len(robust_result.evaluation_history_avg) + 1)
for label, hist, color in [
    ("nominal", robust_result.evaluation_history_nominal, style.COLOR_CYCLE[0]),
    ("eroded", robust_result.evaluation_history_eroded, style.COLOR_CYCLE[1]),
    ("dilated", robust_result.evaluation_history_dilated, style.COLOR_CYCLE[2]),
    ("average (optimized)", robust_result.evaluation_history_avg, style.COLOR_STEEL),
]:
    ax.plot(iterations, hist, "-o", ms=3, label=label, color=color)
for stage_end in range(iters_per_stage, len(iterations), iters_per_stage):
    ax.axvline(stage_end + 0.5, color=style.COLOR_REFERENCE, linestyle="--", linewidth=1, alpha=0.8)
ax.set_xlabel("NLopt iteration")
ax.set_ylabel("objective J")
ax.set_title("Robust adjoint optimization convergence (beta-continuation stages)")
ax.legend()
plt.show()

## 13. Electromagnetic field visualization -- final design (nominal rendering)

In [ ]:
result = btr.simulate_baseline_compact(params, weights=robust_result.final_weights_binarized["nominal"])
fs = result.field_snapshot
print(f"dominant field: {fs.component}")

fig, (ax_eps, ax_field) = plt.subplots(1, 2, figsize=(13, 6))
viz.plot_permittivity(result.permittivity.eps, result.permittivity.extent_um, ax=ax_eps,
                       ports=ports, pml_um=params["dpml_um"], cmap=style.CMAP_PERMITTIVITY_OPTIMIZED)
ax_eps.set_title("Robust design -- nominal rendering")
viz.plot_field(fs.field, fs.eps, fs.extent_um, component=fs.component, ax=ax_field, pml_um=params["dpml_um"])
fig.tight_layout()
plt.show()

> **STOP -- inspect the field before trusting the S-parameters below.**

## 14. S-parameters -- the physically authoritative measurement (nominal sub-design)

In [ ]:
for key in ["11", "12", "21", "22"]:
    s = result.s_matrix[key]
    i_mid = len(s) // 2
    print(f"S{key}: |S|^2={np.abs(s[i_mid])**2:.4f}  phase={np.angle(s[i_mid]):.3f} rad  "
          f"(at wavelength={result.wavelengths_um[i_mid]:.4f} um)")

s21p_mid = np.abs(result.s_matrix["21"][len(result.s_matrix["21"]) // 2]) ** 2
print(f"\nInsertion loss at mid-band: {-10 * np.log10(s21p_mid):.3f} dB")

fig = viz.plot_sparams(result.wavelengths_um, result.s_matrix, pairs=("11", "21", "22", "12"))
plt.show()

## 15. Sanity checks

In [ ]:
ENERGY_TOL = 0.10
RECIPROCITY_TOL = 0.25  # loosened for this freeform design's lack of arm-swap symmetry,
                         # same rationale as bend_topopt.py's own tolerance.
PASSIVITY_TOL = 0.02
MAX_LOSS_FRACTION = 0.2

validation = checks.run_all_checks(
    result.s_matrix,
    energy_tol=ENERGY_TOL,
    reciprocity_tol=RECIPROCITY_TOL,
    passivity_tol=PASSIVITY_TOL,
    max_loss_fraction=MAX_LOSS_FRACTION,
)
print(json.dumps(validation, indent=2))
assert validation["passed"], "Robust design failed sanity checks -- do not save or use this result."
print("\nRobust design validated.")

## 16. Fabrication-bias validation of the robust design + 6-panel field/eps visualization

The whole point of Sections 10-15: does the resulting design actually show a
SMALLER transmission drop under ±10nm than the baseline (Section 8)? The
`final_weights_continuous` eroded/dilated variants came straight out of the
optimizer's own `eta_e`/`eta_d` mapping -- no re-filtering/upsampling
approximation needed. Each of the 3 `simulate_baseline_compact` calls below
already captures a full field snapshot (`capture_dft=True` on the "o1"
launch direction) as part of the honest transmission measurement -- the
6-panel grid below reuses those snapshots directly rather than running any
additional FDTD.

In [ ]:
robust_fab_tolerance = {}
robust_results_by_label = {}
for label in ("nominal", "eroded", "dilated"):
    res_params = {**params, "resolution": chosen_resolution}
    r = btr.simulate_baseline_compact(res_params, weights=robust_result.final_weights_continuous[label])
    robust_results_by_label[label] = r
    i_mid = len(r.s_matrix["21"]) // 2
    robust_fab_tolerance[label] = np.abs(r.s_matrix["21"][i_mid]) ** 2
    print(f"{label}: T={robust_fab_tolerance[label]:.4f}  ({-10*np.log10(robust_fab_tolerance[label]):.3f} dB)")

max_delta_robust = max(abs(robust_fab_tolerance["dilated"] - robust_fab_tolerance["nominal"]),
                        abs(robust_fab_tolerance["eroded"] - robust_fab_tolerance["nominal"]))
print(f"max|delta T| = {max_delta_robust:.4f}")

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(12, 15))
for row, label in enumerate(("nominal", "eroded", "dilated")):
    r = robust_results_by_label[label]
    fs_r = r.field_snapshot
    viz.plot_permittivity(r.permittivity.eps, r.permittivity.extent_um, ax=axes[row, 0],
                           pml_um=params["dpml_um"], cmap=style.CMAP_PERMITTIVITY_OPTIMIZED)
    axes[row, 0].set_title(f"{label} -- permittivity")
    viz.plot_field(fs_r.field, fs_r.eps, fs_r.extent_um, component=fs_r.component, ax=axes[row, 1],
                   pml_um=params["dpml_um"])
    axes[row, 1].set_title(f"{label} -- Re({fs_r.component})")
fig.tight_layout()
plt.show()

## 17. Before vs. after comparison

In [ ]:
labels = ["eroded (-10nm)", "nominal", "dilated (+10nm)"]
baseline_vals = [baseline_fab_tolerance["eroded"], baseline_fab_tolerance["nominal"], baseline_fab_tolerance["dilated"]]
robust_vals = [robust_fab_tolerance["eroded"], robust_fab_tolerance["nominal"], robust_fab_tolerance["dilated"]]

fig, ax = plt.subplots(figsize=(7, 4.5))
x = np.arange(len(labels))
width = 0.35
ax.bar(x - width / 2, baseline_vals, width, label="baseline (best of 5 random seeds)", color=style.COLOR_REFERENCE)
ax.bar(x + width / 2, robust_vals, width, label="this notebook (3-way averaged objective)", color=style.COLOR_SEAGREEN)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel(r"transmission $|S_{21}|^2$")
ax.set_title("Fabrication-bias tolerance: before vs. after")
ax.legend()
plt.show()

print(f"baseline design:     max|delta T| = {max_delta_baseline:.4f}")
print(f"this robust design:  max|delta T| = {max_delta_robust:.4f}")
improvement = (max_delta_baseline - max_delta_robust) / max_delta_baseline if max_delta_baseline > 0 else float("nan")
print(f"change: {improvement:+.1%} {'reduction' if improvement > 0 else 'INCREASE'} in worst-case "
      f"fabrication-bias sensitivity vs. the baseline")

## 18. Save artifacts

In [ ]:
artifact_stem = "../data/sparams/bend_topopt_robust/baseline/bend_topopt_robust_baseline"

metadata = {
    "component": "bend_topopt_robust",
    "geometry_params": {
        "wg_width_um": result.sim_params["wg_width_um"],
        "design_region_x_um": result.sim_params["design_region_x_um"],
        "design_region_y_um": result.sim_params["design_region_y_um"],
        "design_grid_n": result.sim_params["design_grid_n"],
        "margin_um": result.sim_params["margin_um"],
        "arm_lead_um": result.sim_params["arm_lead_um"],
        "filter_radius_um": result.sim_params["filter_radius_um"],
        "beta_schedule": result.sim_params["beta_schedule"],
        "eta_i": result.sim_params["eta_i"],
        "eta_e": result.sim_params["eta_e"],
        "eta_d": result.sim_params["eta_d"],
        "target_bias_um": result.sim_params["target_bias_um"],
    },
    "material_params": {
        "core_index": result.sim_params["core_index"],
        "clad_index": result.sim_params["clad_index"],
    },
    "polarization": f"TE ({result.field_snapshot.component} out-of-plane, confirmed dominant -- see Section 13)",
    "meep_resolution": result.sim_params["resolution"],
    "simulation_params": {
        k: result.sim_params[k]
        for k in ["wl_min_um", "wl_max_um", "n_freq", "dpml_um",
                  "wavelength_um", "iters_per_stage", "adjoint_minimum_run_time"]
    },
    "optimization": {
        "final_objective_avg": float(robust_result.evaluation_history_avg[-1]),
        "final_objective_nominal": float(robust_result.evaluation_history_nominal[-1]),
        "final_objective_eroded": float(robust_result.evaluation_history_eroded[-1]),
        "final_objective_dilated": float(robust_result.evaluation_history_dilated[-1]),
        "n_iterations": len(robust_result.evaluation_history_avg),
        "fill_fraction": float(robust_result.final_weights_binarized["nominal"].mean()),
    },
    "meep_version": result.sim_params["meep_version"],
    "python_version": result.sim_params["python_version"],
    "creation_date": result.sim_params["creation_date"],
    "validation": validation,
}

sparams.save_artifact(
    artifact_stem, result.wavelengths_um, result.freqs,
    result.s_matrix, result.port_names, metadata,
)
print(f"Saved artifact: {artifact_stem}.npz / .json")

np.save("../data/sparams/bend_topopt_robust/baseline/bend_topopt_robust_density_baseline_best.npy", baseline_mask)
print("Saved baseline (best-of-5) density map")

for label in ("nominal", "eroded", "dilated"):
    density_path = f"../data/sparams/bend_topopt_robust/baseline/bend_topopt_robust_density_{label}.npy"
    np.save(density_path, robust_result.final_weights_binarized[label])
    print(f"Saved density map ({label}): {density_path}")

convergence_path = "../data/sparams/bend_topopt_robust/baseline/bend_topopt_robust_convergence.npz"
np.savez(
    convergence_path,
    evaluation_history_nominal=robust_result.evaluation_history_nominal,
    evaluation_history_eroded=robust_result.evaluation_history_eroded,
    evaluation_history_dilated=robust_result.evaluation_history_dilated,
    evaluation_history_avg=robust_result.evaluation_history_avg,
)
print(f"Saved convergence history: {convergence_path}")

## 19. Design point

In [ ]:
design_point_path = "../data/design_points/bend_topopt_robust.yaml"

design_points.save_design_point(design_point_path, {
    "component": "bend_topopt_robust",
    "parameters": {**metadata["geometry_params"], **metadata["material_params"]},
    "source_artifact": "data/sparams/bend_topopt_robust/baseline/bend_topopt_robust_baseline",
    "provenance": {
        "method": "robust_adjoint_topology_optimization_3way_averaged_objective_compact_domain",
        "based_on": "bend_topopt",
        "date": metadata["creation_date"],
        "meep_version": metadata["meep_version"],
    },
    "validation": validation,
    "robustness": {
        "calibration": {
            "method": (
                "Straight-edge test pattern on the design_grid_n grid, conic_filter + "
                "tanh_projection at candidate eta, rendered through a real mp.MaterialGrid "
                "at calibration_resolution=200px/um (structure-only, no timestepping); "
                "boundary located at the core/clad epsilon midpoint. See "
                "bend_topopt_robust.calibrate_eta_bias()."
            ),
            "beta_used_for_calibration": 32.0,
            "eta_bias_check_beta4_vs_beta32": {
                "eta_e_beta4": float(eta_e_beta4), "eta_d_beta4": float(eta_d_beta4),
                "eta_e_beta32": float(eta_e), "eta_d_beta32": float(eta_d),
            },
        },
        "baseline_multistart": {
            "seeds": seeds,
            "transmission_by_seed": {str(s): float(transmissions[s]) for s in seeds},
            "best_seed": int(best_seed),
            "gate_threshold": float(params["baseline_transmission_gate"]),
            "gate_passed": True,
        },
        "fabrication_transmission_comparison": {
            "baseline_best_of_5_random_start_compact_domain": {
                "transmission_nominal": float(baseline_fab_tolerance["nominal"]),
                "transmission_plus_bias_dilated": float(baseline_fab_tolerance["dilated"]),
                "transmission_minus_bias_eroded": float(baseline_fab_tolerance["eroded"]),
                "max_abs_delta": float(max_delta_baseline),
            },
            "this_robust_design": {
                "transmission_nominal": float(robust_fab_tolerance["nominal"]),
                "transmission_plus_bias_dilated": float(robust_fab_tolerance["dilated"]),
                "transmission_minus_bias_eroded": float(robust_fab_tolerance["eroded"]),
                "max_abs_delta": float(max_delta_robust),
            },
            "relative_improvement": float(improvement),
        },
        "resolution_convergence_check": {
            "resolutions_px_per_um": [20, 40],
            "transmission_delta_at_resolution": {"20": float(delta_20), "40": float(delta_40)},
            "relative_change": float(relative_change),
            "converged": bool(resolution_converged),
            "resolution_used_for_reported_numbers": int(chosen_resolution),
        },
        "density_maps": {
            "baseline_best_of_5": "data/sparams/bend_topopt_robust/baseline/bend_topopt_robust_density_baseline_best.npy",
            "nominal": "data/sparams/bend_topopt_robust/baseline/bend_topopt_robust_density_nominal.npy",
            "eroded": "data/sparams/bend_topopt_robust/baseline/bend_topopt_robust_density_eroded.npy",
            "dilated": "data/sparams/bend_topopt_robust/baseline/bend_topopt_robust_density_dilated.npy",
        },
    },
    "rationale": (
        f"Adjoint-optimized over {len(robust_result.evaluation_history_avg)} NLopt MMA "
        f"iterations on a compact ({dom['cell_x_um']:.2f}x{dom['cell_y_um']:.2f}um) domain and "
        f"a {params['design_region_x_um']}x{params['design_region_y_um']}um design region, "
        f"maximizing the average of three objective renderings of the same raw design "
        f"variables every iteration -- nominal (eta_i={params['eta_i']}), eroded "
        f"(eta_e={eta_e:.5f}, -{1000*params['target_bias_um']:.0f}nm), and dilated "
        f"(eta_d={eta_d:.5f}, +{1000*params['target_bias_um']:.0f}nm). Compared against a "
        f"fresh, non-Euler-warm-started baseline (best of {len(seeds)} random seeds, "
        f"T21={transmissions[best_seed]:.4f}, seed={best_seed}) on this SAME compact domain, "
        f"not against notebook 04's own (Euler-warm-started, and therefore already somewhat "
        f"robust) design. Headline result: the baseline's max|delta T|={max_delta_baseline:.4f} "
        f"transmission swing under +/-{1000*params['target_bias_um']:.0f}nm became "
        f"{max_delta_robust:.4f} ({improvement:+.1%} change) after the averaged-objective "
        f"optimization. Validated by an independent two-port FDTD measurement of the nominal "
        f"sub-design (not the optimizer's own objective)."
    ),
})
print(f"Saved design point: {design_point_path}")

## 20. Bend-topopt-robust SAX model

Not exercised here (this process already imported meep) -- verified
separately in a clean, meep-free process.

In [ ]:
with open("../src/pic_toolkit/models/bend_topopt_robust.py") as f:
    print(f.read())

---

## Summary

A compact, `bend.py`-style domain (~36% smaller cell area than the earlier
revision's inherited symmetric-clearance box) was built for a 3x3um design
region. A non-Euler baseline was found by a 5-random-seed search, gated at
>90% transmission, and its own ±10nm fabrication-bias sensitivity was
measured honestly. A new adjoint optimization then averaged the objective
over nominal/eroded/dilated renderings of the same design variables every
iteration, producing a design tolerant to that bias by construction. The
result was re-measured with the toolkit's standard two-port method, cached
as `bend_topopt_robust`, and can be loaded meep-free via
`from pic_toolkit.models.bend_topopt_robust import bend_topopt_robust`. See
`docs/simulation_settings_record.md`'s `bend_topopt_robust.py` section for
the full measurement history behind every parameter above.